# Graded Lab: Agentic Workflows

In this lab, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [23]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [24]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [25]:
CLIENT = Client()

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:o4-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [26]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1, 
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [27]:
# Test your code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


In [28]:
planner_agent(topic = "why people catch cold when weather changes?")

['Instruct Research Agent to perform a comprehensive search on academic databases and reputable websites for studies linking weather changes to common cold incidence',
 'Instruct Research Agent to extract and compile epidemiological data on cold incidence correlated with temperature and humidity fluctuations',
 'Instruct Research Agent to gather information on physiological mechanisms by which temperature variability impacts human immune response',
 'Instruct Research Agent to collect virology insights on common cold viruses, including environmental stability and transmission under changing weather conditions',
 'Instruct Writer Agent to draft a structured outline covering introduction, epidemiology findings, physiological mechanisms, virology insights, and conclusions',
 'Instruct Writer Agent to expand the outline into a detailed first draft with proper citations and section headings',
 'Instruct Editor Agent to review the first draft for scientific accuracy, logical coherence, and c

## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [29]:
#     prompt_ = f"""
# You are a fashion market research agent tasked with preparing a trend analysis for a summer sunglasses campaign.

# Your goal:
# 1. Explore current fashion trends related to sunglasses using web search.
# 2. Review the internal product catalog to identify items that align with those trends.
# 3. Recommend one or more products from the catalog that best match emerging trends.
# 4. If needed, today date is {datetime.now().strftime("%Y-%m-%d")}.

# You can call the following tools:
# - tavily_search_tool: to discover external web trends.
# - product_catalog_tool: to inspect the internal sunglasses catalog.

# Once your analysis is complete, summarize:
# - The top 2–3 trends you found.
# - The product(s) from the catalog that fit these trends.
# - A justification of why they are a good fit for the summer campaign.
# """
#     messages = [{"role": "user", "content": prompt_}]
#     tools_ = tools.get_available_tools()

In [30]:
# GRADED FUNCTION: research_agent

def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = f"""
    You are a Research Assistant agent your task is to do reasearch on topics given by user, gathering 
    information from internet using given tools.
    
    Your Goal:
    1. Explore the Current Research Papers available online on the given topic
    2. Carefully read all the papers regarding that topic
    3. return the gather information from revelevant papers
    4. If needed, today date is {current_time}.
    
    You can call following tools:
    -  arxiv_tool: to find academic papers.
    - tavily_tool: for general web searches.
    - wikipedia_tool: for accessing encyclopedic knowledge.
    
    User Topic:
    {task}
    """
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [research_tools.arxiv_search_tool, research_tools.tavily_search_tool, research_tools.wikipedia_search_tool]
    
    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(  
        # Set the model
        model=model,
        # Pass in the messages. You already defined this!
        messages=messages,
        # Pass in the tools list. You already defined this!
        tools=tools,
        # Set the LLM to automatically choose the tools
        tool_choice="auto",
        # Set the max turns to 6
        max_turns=6
    )  
    
    ### END CODE HERE ###

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [31]:
# Test your code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 Please provide me with the specific topic or query for which you would like me to find and summarize research papers.
🔍 Research Agent
✅ Output:
 Please provide the topic you're interested in exploring, so I can begin researching the relevant papers for you.
 All tests passed!


## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [32]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt =f"""you are a specialized writing agent which writes well structured academic or technical draft on the given 
    information provide by user"""

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role":"system", "content":system_prompt}

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role":"user", "content":task}

    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [33]:
# Test your code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [34]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    # Create the system prompt.
    # This should assign the LLM the role of an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    system_prompt = "you are best editor agent you task is reflect and critiquing and improve initial drafts given by user"
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role":"system","content":system_prompt}
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role":"user","content":task}
    
    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    ### END CODE HERE ###
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [35]:
# Test your code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

👉 **Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [36]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [37]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [38]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: Search arXiv and Wikipedia for foundational papers on ensemble Kalman filter time series forecasting
🔍 Research Agent
✅ Output:
 I encountered a timeout error when attempting to retrieve research papers from arXiv. However, I successfully retrieved information from Wikipedia about the ensemble Kalman filter (EnKF). Here's a brief overview:

### Ensemble Kalman Filter
- **Definition**: The Ensemble Kalman Filter (EnKF) is a recursive filter designed for problems with a large number of variables, making it suitable for discretizations of partial differential equations in geophysical models. 
- **Origin and Application**: EnKF was developed as a version of the Kalman filter for large problems where the covariance matrix is replaced by the sample covariance. It now serves as a significant data assimilation component in ensemble forecasting.
- **Relation to Particle Filter**: Although related to the particle filter, where a 


🛠️ Executing with agent: `research_agent` on task: Extract and summarize key algorithms, mathematical formulations, and performance results from each paper
🔍 Research Agent
✅ Output:
 Here's a detailed summary of the key algorithms, mathematical formulations, and performance results from the selected papers on Ensemble Kalman Filters (EnKF) and related applications:

1. **[Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference](https://arxiv.org/abs/2312.05910v5)**
   - **Key Algorithms and Formulations**: The paper integrates EnKF into Gaussian process state-space models (GPSSMs) for non-mean-field variational inference. This novel approach addresses issues related to training effort and inference accuracy. It utilizes EnKF to approximate the posterior distribution of latent states, eliminating the need for extensive parameterization, and facilitates interpretable approximation of the evidence lower bound (ELBO).
   - **Performance**: The propose


🛠️ Executing with agent: `research_agent` on task: Summarize benchmark comparisons and typical use cases of ensemble Kalman filters in time series forecasting
🔍 Research Agent
✅ Output:
 Here is a summary of the information collected on the benchmark comparisons and typical use cases of Ensemble Kalman Filters (EnKF) in time series forecasting:

### Benchmark Comparisons in Time Series Forecasting
1. **LLM-Mixer for Time Series Forecasting**:
   - **Summary**: The LLM-Mixer framework combines multiscale time-series decomposition with pre-trained Large Language Models (LLMs) to improve forecasting accuracy. This method captures both short-term fluctuations and long-term trends, outperforming state-of-the-art models across various forecasting horizons.
   - **Performance**: Exhibits competitive performance in both multivariate and univariate datasets by leveraging the combination of multiscale analysis and LLMs.
   - [PDF Link](https://arxiv.org/pdf/2410.11674v2)

2. **Probabilistic Hie

Here is a summary of the information collected on the benchmark comparisons and typical use cases of Ensemble Kalman Filters (EnKF) in time series forecasting:

### Benchmark Comparisons in Time Series Forecasting
1. **LLM-Mixer for Time Series Forecasting**:
   - **Summary**: The LLM-Mixer framework combines multiscale time-series decomposition with pre-trained Large Language Models (LLMs) to improve forecasting accuracy. This method captures both short-term fluctuations and long-term trends, outperforming state-of-the-art models across various forecasting horizons.
   - **Performance**: Exhibits competitive performance in both multivariate and univariate datasets by leveraging the combination of multiscale analysis and LLMs.
   - [PDF Link](https://arxiv.org/pdf/2410.11674v2)

2. **Probabilistic Hierarchical Forecasting**:
   - **Summary**: Uses a Deep Poisson Mixture Network (DPMN) for accurate and coherent probabilistic forecasts in hierarchical time series structures.
   - **Performance**: Achieves notable improvements in hierarchical coherence and probabilistic forecasts, as evidenced by CRPS improvements on datasets like Australian domestic tourism data.
   - [PDF Link](https://arxiv.org/pdf/2110.13179v8)

3. **SCINet for Time Series Modeling**:
   - **Summary**: SCINet is proposed for modeling time series with complex temporal dynamics using a deep neural network architecture that utilizes sample convolution and interaction.
   - **Performance**: Shows significant forecasting accuracy improvements over traditional convolutional and Transformer-based models.
   - [PDF Link](https://arxiv.org/pdf/2106.09305v3)

### Wikipedia Summary of Kalman Filter
The Kalman filter, named after Rudolf E. Kálmán, is widely used in statistics and control theory as an algorithm that enhances the accuracy of estimates by using a series of observed measurements over time. It is particularly useful in dynamic applications like guidance, navigation, and control of vehicles, making it applicable to aircraft and spacecraft navigation and positioning systems. The EnKF, as a derived version for larger state-space problems, continues to find applications in various fields requiring precise data assimilation and time series forecasting.

For further reading on Kalman filters, visit the [Wikipedia page](https://en.wikipedia.org/wiki/Kalman_filter).

The search using Tavily timed out, but the information gathered from arXiv and Wikipedia provides a comprehensive overview of the current research and comparisons in the field of Ensemble Kalman Filters applied to time series forecasting. If you need further details or specific studies, let me know!

## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>